El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

Significado de las columnas de los datos:

- DateCrawled — fecha en la que se descargó el perfil de la base de datos
- VehicleType — tipo de carrocería del vehículo
- RegistrationYear — año de matriculación del vehículo
- Gearbox — tipo de caja de cambios
- Power — potencia (CV)
- Model — modelo del vehículo
- Mileage — kilometraje (medido en km de acuerdo con las especificidades regionales del conjunto de datos)
- RegistrationMonth — mes de matriculación del vehículo
- FuelType — tipo de combustible
- Brand — marca del vehículo
- NotRepaired — vehículo con o sin reparación
- DateCreated — fecha de creación del perfil
- NumberOfPictures — número de fotos del vehículo
- PostalCode — código postal del propietario del perfil (usuario)
- LastSeen — fecha de la última vez que el usuario estuvo activo

### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import (OneHotEncoder, 
                                   StandardScaler)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import joblib


### Preparación de datos

#### Cargar datos

In [ ]:
nombre_de_archivo = "car_data.csv"
ruta_a_datasets = "datasets"
ruta_completa = os.path.join(ruta_a_datasets, nombre_de_archivo)

In [ ]:
df = pd.read_csv(ruta_completa)

#### Análisis exploratorio de datos (EDA)

In [ ]:
# Ver la información del DataFrame
df.info()

In [ ]:
df.describe()

##### Price

In [ ]:
# Se observan muchos autos con precios inferiores a 30 dólares. Se decide explorar un poco más
df[df["Price"] < 500] # Explorar modelos de autos con precio menor a 500 dólares

In [ ]:
# Se aprecian autos cuyos precios no están acorde a la realidad; ejemplo el Polo de Volkswagen tiene un precio de 300 dólares; aunque tenga 150 mil millas;
# no tiene reparaciones; valorado en el mercado entre 2000 a 3000 dólares (búsqueda de internet). 
# Se decide eliminar autos con precio menor a 500 dólares; para mantener el modelo simple y no tener que hacer una limpieza más profunda de los datos.
df_filtrado = df[df["Price"] >= 500]

In [ ]:
len(df_filtrado)

##### VehicleType

In [ ]:
# Explorar tipos de vehículos.
df_filtrado[df_filtrado["VehicleType"].isna()]

In [ ]:
# Se deciden eliminar todos los vehículos que se desconoce su tipo; porque no se puede inferir el tipo de vehículo y no se puede reemplazar por un valor genérico.
# Además, se sabe que el tipo de vehículo es importante para el precio. Por ejemplo; un SUV es más caro que un sedán.
df_filtrado = df_filtrado[df_filtrado["VehicleType"].notna()]

In [ ]:
len(df_filtrado)

##### Registration Year

In [ ]:
# En la descripción se aprecia que el año de registro tiene un mínimo de 1000 y un máximo de 9999, lo cual no es realista. 
# Se decide eliminar autos cuyo registro sea inferior a 1990 y superior a 2026.
# Los precios de autos varian mucho; los autos viejos y de colección son autos muy caros; al igual que los autos nuevos.
# Para mantener el modelo simple, solo se dejaran autos "modernos" y se eliminarán los autos antíguos; porque se desconoce si son de colección o solo meten ruido al modelo.
df_filtrado = df_filtrado[(df_filtrado["RegistrationYear"] >= 1990) & (df_filtrado["RegistrationYear"] <= 2026)]
df_filtrado["RegistrationYear"].unique()

In [ ]:
len(df_filtrado)

##### Gearbox

In [ ]:
# Explorar Gearbox
df_filtrado["Gearbox"].unique()

In [ ]:
# Ahora se reemplazaran los valores ausentes de la columna "Gearbox" por "manual". 
df_filtrado["Gearbox"] = df_filtrado["Gearbox"].fillna("manual") # Reemplazar valores ausentes por "manual"

In [ ]:
len(df_filtrado)

##### Power

In [ ]:
# Así mismo se observa que hay autos que tiene una potencia (Power) de 0, esto tampoco es realista; se exploran los tipos de autos que tienen potencia menor a 30 
# para decidir si eliminar o reemplazar.
df_filtrado[(df_filtrado["Power"] <30)]["Model"].unique()

In [ ]:
# Se decide eliminar estos registros; hay todo tipo de vehículos en esta lista, carros de golf, camionetas, autos de lujo. Mantener estos registros afectará al modelo.
# En internet se consiguió que los autos promedios tienen una potencia promedio de 100 CV (autos); sin embargo hay autos como el Bettle (Volkswagen) que tiene una potencia de 75 CV, por lo que se decide eliminar autos con potencia de 35 CV.
# Esto coincide con el promedio obtenido en la descripción del DataFrame.
# Se decide dejar solamente autos con potencia mayor o igual a 30 CV
df_filtrado = df_filtrado[df_filtrado["Power"] >= 30]

In [ ]:
len(df_filtrado)

##### Model

In [ ]:
df_filtrado["Model"].unique()

In [ ]:
df_filtrado[df_filtrado["Model"].isna()]

In [ ]:
# Se decide eliminar autos cuyo modelo sea desconocido. El modelo influye mucho en el precio
df_filtrado = df_filtrado[df_filtrado["Model"].notna()]

In [ ]:
len(df_filtrado)

##### Brand

In [ ]:
df_filtrado["Brand"].unique()

In [ ]:
df_filtrado[df_filtrado["Brand"].isna()]

##### Fuel Type

In [ ]:
# Explorar FuelType
df_filtrado["FuelType"].unique()

In [ ]:
# Explorar FuelType
print("Tipos de combustible en el registro de autos:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["VehicleType"].unique())
print("")
print("Años de registro únicos de autos con FuelType nulo:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["RegistrationYear"].unique())
print("")    
print("Cantidad de autos con FuelType nulo:")
print(len(df_filtrado[df_filtrado["FuelType"].isnull()]))

In [ ]:
# Se decide eliminar estos registros; ya que es posible que hayan autos híbridos o eléctricos dentro del registro.
df_filtrado = df_filtrado[df_filtrado["FuelType"].notna()] # El tipo de combustible afecta un poco el precio, pero son pocos registros

In [ ]:
len(df_filtrado)

##### Repairs

In [ ]:
df_filtrado[df_filtrado["NotRepaired"].isna()]

In [ ]:
df_filtrado["NotRepaired"] = df_filtrado["NotRepaired"].fillna("no") # Reemplazar valores ausentes por "No"
df_filtrado["NotRepaired"].unique()

In [ ]:
len(df_filtrado)

In [ ]:
# Ahora con los datos filtrados; se decide dejar unicamente las columnas que aporten información útil para el modelo:
# Price; VehicleType; RegistrationYear; Gearbox; Power; Mileage; FuelType; Brand; NotRepaired

In [ ]:
df_filtrado.info()

In [ ]:
# Nota; aunque es posible que el código postal pueda influir en el precio del auto; se decide eliminar esta columna como entrada del modelo.
# Lo ideal sería segmentar autos con mismas características para cada código postal; luego habría que hacer una t de student para evaluar si existe
# diferencia estadísticamente significativa entre la media de los precios. 
# Otra forma sería hacer un análisis bootstrapping para estimar la media y la desviación estándar de los precios para cada código postal; 
# luego comparar los intervalos de confianza de cada código postal. De hecho, esto sería lo mejor porque no se asume que los precios 
# de los autos tengan una distribución normal. Y además, incorporla la variabilidad de todos los precios de autos sin centrarse en un tipo
# en específico.
 
data = df_filtrado[["Price", "VehicleType", "RegistrationYear", "Gearbox", "Power", "Mileage", "FuelType", "Brand", "NotRepaired"]]

In [ ]:
data.info()

##### Conclusion del EDA y limpieza de datos

- Se filtraron los datos COMPLETAR

## Entrenamiento del modelo 

### Preprocesado

In [ ]:
# Explicación del procedimiento:
# 1. Dividir los datos en target y features de la siguiente forma:
# Target = Price
# Features = VehicleType, RegistrationYear, Gearbox, Power, Mileage, FuelType, Brand, NotRepaired

# 2. Dividir los datos en datos de prueba y de entrenamiento en proporción 75% entrenamiento y 25% prueba 
# NOTA: usaré train_test_split en lugar de cross_val_score porque voy a entrenar muchos modelos y el poder de computo es limitado.

# 3. Crear dos sets de datos:
# 3.1. Datos SIN encoding y estandarizados para entrenar el modelo CatBoost y LightGBM
# 3.2. Datos CON encoding y estandarizados para entrenar todos los demás modelos.


In [ ]:
# 1. Datos sin encoding para entrenar el modelo CatBoost y LightGBM
features = data.drop("Price", axis=1)
target = data["Price"]

In [ ]:
# 2. Dividir los datos en datos de prueba y de entrenamiento en proporción 75% entrenamiento y 25% prueba
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.25, random_state=12345)

In [ ]:
# 3. Crear dos sets de datos:

# 3.1. Datos SIN encoding y estandarizados para entrenar el modelo CatBoost y LightGBM
X_train_no_encoding = X_train.copy()
X_test_no_encoding = X_test.copy()

In [ ]:
#3.2# Separar variables categóricas y numéricas
cat_cols = ["VehicleType",
            "Gearbox",
            "FuelType",
            "Brand",
            "NotRepaired"]

num_cols = ["RegistrationYear",
            "Power",
            "Mileage"]

# Crear el preprocesador
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, cat_cols),
        ("num", scaler, num_cols)
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [ ]:
# Los resultados de correr cada modelo se guardarán en el siguiente diccionario:
results = {
    "datasets": {
        "X_train_no_encoding": X_train_no_encoding,
        "X_test_no_encoding": X_test_no_encoding,
        "X_train_encoded": X_train_encoded,
        "X_test_encoded": X_test_encoded,
        "y_train": y_train,
        "y_test": y_test
    },

    "Lineal Regression": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "Decision Tree": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "best_depth": None,
        "best_leaf_size": None,
        "y_pred": None,
        "model": None
    },
    "Random Forest": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "best_depth": None,
        "best_leaf_size": None,
        "best_estimators": None,
        "y_pred": None,
        "model": None
    },
    "XGBoost": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "LightGBM": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "CatBoost": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    }
}

### Regresion lineal

In [ ]:
model = LinearRegression()
model.fit(X_train_encoded, y_train)
y_pred = model.predict(X_test_encoded)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)**0.5
mae = mean_absolute_error(y_test, y_pred)

print("******************************** LINEAR REGRESSION ********************************")
print("R2 score:", r2)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("")

results["Lineal Regression"]["r2_score"] = r2
results["Lineal Regression"]["rmse"] = rmse
results["Lineal Regression"]["mae"] = mae
results["Lineal Regression"]["y_pred"] = y_pred
results["Lineal Regression"]["model"] = model


###  Árbol de decisión 

In [ ]:
best_depth = 0
best_leaf_size = 0
best_r2 = float('-inf')  # Initialize best R2 score to negative infinity
best_rmse = float('inf')  # Initialize best RMSE to positive infinity

depths = [5, 7, 10, 15]
leaf_sizes = [5, 10, 20, 50]
for depth in depths:
    for leaf_size in leaf_sizes:
        model = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf_size, random_state=12345)
        model.fit(X_train_encoded, y_train)
        y_pred = model.predict(X_test_encoded)
        r2 = r2_score(y_test, y_pred)
        rmse = mean_squared_error(y_test, y_pred)**0.5
        mae = mean_absolute_error(y_test, y_pred)
        print(f"Depth: {depth}, Leaf Size: {leaf_size}")
        print("R2 score:", r2)
        print("Root Mean Squared Error:", rmse)
        print("Mean Absolute Error:", mae)
        print("-----------------------------")
        if r2 > best_r2:
            best_r2 = r2
            best_rmse = rmse
            best_mae = mae
            best_depth = depth
            best_leaf_size = leaf_size
            y_pred_best = y_pred
            best_model = model

print("******************************** DECISION TREE BEST MODEL ********************************")
print("Best Depth:", best_depth)
print("Best Leaf Size:", best_leaf_size)
print("Best R2 Score:", best_r2)
print("Best RMSE:", best_rmse)
print("Best MAE:", best_mae)
print("")

results["Decision Tree"]["r2_score"] = best_r2
results["Decision Tree"]["rmse"] = best_rmse
results["Decision Tree"]["mae"] = best_mae
results["Decision Tree"]["best_depth"] = best_depth
results["Decision Tree"]["best_leaf_size"] = best_leaf_size
results["Decision Tree"]["y_pred"] = y_pred_best
results["Decision Tree"]["model"] = best_model

### Bosques aleatorios (Random Forest)

In [ ]:
depths = [5, 7, 10, 15]
leaf_sizes = [5, 10, 20, 50]
n_estimators = [50, 100, 200]

best_r2 = float('-inf')  # Initialize best R2 score to negative infinity
best_rmse = float('inf')  # Initialize best RMSE to positive infinity


for depth in depths:
    for leaf_size in leaf_sizes:
        for n_estimator in n_estimators:
            model = RandomForestRegressor(n_estimators=n_estimator, max_depth=depth, min_samples_leaf=leaf_size, random_state=12345)
            model.fit(X_train_encoded, y_train)
            y_pred = model.predict(X_test_encoded)
            r2 = r2_score(y_test, y_pred)
            rmse = mean_squared_error(y_test, y_pred)**0.5
            mae = mean_absolute_error(y_test, y_pred)
            print(f"Depth: {depth}, Leaf Size: {leaf_size}, N Estimators: {n_estimator}")
            print("R2 score:", r2)
            print("Root Mean Squared Error:", rmse)
            print("Mean Absolute Error:", mae)
            print("-----------------------------")
            if r2 > best_r2:
                best_r2 = r2
                best_rmse = rmse
                best_mae = mae
                best_depth = depth
                best_leaf_size = leaf_size
                best_n_estimators = n_estimator
                y_pred_best = y_pred
                best_model = model

print("******************************** RANDOM FOREST BEST MODEL ********************************")
print("Best Depth:", best_depth)
print("Best Leaf Size:", best_leaf_size)
print("Best N Estimators:", best_n_estimators)
print("Best R2 Score:", best_r2)
print("Best RMSE:", best_rmse)
print("Best MAE:", best_mae)
print("")

results["Random Forest"]["r2_score"] = best_r2
results["Random Forest"]["rmse"] = best_rmse
results["Random Forest"]["mae"] = best_mae
results["Random Forest"]["best_depth"] = best_depth
results["Random Forest"]["best_leaf_size"] = best_leaf_size
results["Random Forest"]["best_n_estimators"] = best_n_estimators
results["Random Forest"]["y_pred"] = y_pred_best
results["Random Forest"]["model"] = best_model


In [ ]:
joblib.dump(results, "linear_tree_forest_models_results.pkl")

### XGBoost

## Análisis del modelo

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [ ]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [ ]  Se realizó el análisis de velocidad y calidad de los modelos